In [1]:
import pandas as pd
import openpyxl
from datetime import datetime
import os
import re
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO: ALTERE O ANO AQUI
# ═══════════════════════════════════════════════════════════════════════════
ANO = 2024  # ← MUDE PARA 2026, 2025, etc.
# ═══════════════════════════════════════════════════════════════════════════

BASE_PATH = rf"T:\Portugal\D-Trafico\KM BASE\Ano {ANO}"

MONTH_FILES = {
    1:  "01 JANEIRO.xlsx",
    2:  "02 FEVEREIRO.xlsx",
    3:  "03 MARÇO.xlsx",
    4:  "04 ABRIL.xlsx",
    5:  "05 MAIO.xlsx",
    6:  "06 JUNHO.xlsx",
    7:  "07 JULHO.xlsx",
    8:  "08 AGOSTO.xlsx",
    9:  "09 Setembro.xlsx",
    10: "10 Outubro.xlsx",
    11: "11 Novembro.xlsx",
    12: "12 Dezembro.xlsx",
}

SHEETS_COM_TRANSPORTADOR_NA_CELULA = {'Norte'}

TRANSPORTADOR_NORMALIZADO = {
    'TJA':                'TJA',
    'TJA (RÍGIDO)':       'TJA',
    'TJA (SEMI-REBOQUE)': 'TJA',
    'CMTIR':              'CMTIR',
    'CMTIR (RÍGIDO)':     'CMTIR',
    'TFS':                'TFS',
    'TFS (LIGEIRO)':      'TFS',
}

PLATE_PATTERN = re.compile(r'\b([A-Z0-9]{2}-[A-Z0-9]{2}-[A-Z0-9]{2})\b')
PLATE_SPECIAL  = re.compile(r'\b([A-Z]{1,2}-\d{4,5})\b')


def clean_value(value):
    if value is None or value == "":
        return None
    if isinstance(value, str):
        value = value.strip()
        if value.upper() in ["#VALUE!", "#DIV/0!", "PARADO", ""]:
            return None
        try:
            return float(value.replace(',', '.'))
        except:
            return None
    try:
        val = float(value)
        if val == -78:
            return None
        return val if val >= 0 else None
    except:
        return None


def parse_formula(formula):
    if not formula or not isinstance(formula, str):
        return None, None, None
    m = re.match(r'=\w+\*(\d+\.?\d*)', formula)
    if m:
        return 'fixo', 0, float(m.group(1))
    m = re.match(r'=\(\w+-(\d+)\)\*(\d+\.?\d*)', formula)
    if m:
        return 'excedente', int(m.group(1)), float(m.group(2))
    return None, None, None


def extract_tipo_veiculo(text):
    t = text.upper().strip()

    m = re.search(r'CARRO\s+(\d+)\s+PALETES', t)
    if m:
        return f"{m.group(1)} PALETES"

    m = re.search(r'\b(\d+)\s+PALETES\b', t)
    if m:
        return f"{m.group(1)} PALETES"

    m = re.search(r'\(([^)]+)\)', t)
    if m:
        inside = m.group(1).strip()
        if not PLATE_PATTERN.search(inside) and not PLATE_SPECIAL.search(inside):
            return inside

    for kw in ['SEMI-REBOQUE', 'DUPLODECK', 'ELÉTRICO', 'CAPILAR', 'LIGEIRO', 'RÍGIDO']:
        if re.search(rf'\b{re.escape(kw)}\b', t):
            return kw

    return None


def parse_cell_a(text, sheet_name, row_idx, anon_map):
    if text is None:
        return sheet_name, None, None

    text_clean = str(text).strip()
    tipo_veiculo = extract_tipo_veiculo(text_clean)

    plates = PLATE_PATTERN.findall(text_clean.upper())
    if not plates:
        plates = PLATE_SPECIAL.findall(text_clean.upper())

    if plates:
        plate = plates[0]

        if sheet_name in SHEETS_COM_TRANSPORTADOR_NA_CELULA:
            match = PLATE_PATTERN.search(text_clean.upper())
            if not match:
                match = PLATE_SPECIAL.search(text_clean.upper())
            trans_raw = text_clean[:match.start()].strip()
            trans_raw = re.sub(r'\([^)]*\)', '', trans_raw).strip()
            trans_raw = re.sub(r'\s+', ' ', trans_raw).strip()
            trans = TRANSPORTADOR_NORMALIZADO.get(trans_raw.upper(),
                    TRANSPORTADOR_NORMALIZADO.get(trans_raw, trans_raw))
        else:
            trans = TRANSPORTADOR_NORMALIZADO.get(sheet_name, sheet_name)

        return trans, plate, tipo_veiculo

    else:
        if sheet_name in SHEETS_COM_TRANSPORTADOR_NA_CELULA:
            trans_raw = re.sub(r'\([^)]*\)', '', text_clean).strip()
            trans_raw = re.sub(r'\s+', ' ', trans_raw).strip()
            trans = TRANSPORTADOR_NORMALIZADO.get(trans_raw.upper(),
                    TRANSPORTADOR_NORMALIZADO.get(trans_raw, trans_raw))
        else:
            trans = TRANSPORTADOR_NORMALIZADO.get(sheet_name, sheet_name)

        if row_idx not in anon_map:
            prefix = re.sub(r'[^A-Z]', '', trans.upper())[:3]
            n = len(anon_map) + 1
            anon_map[row_idx] = f"{prefix}_N_{n}"

        return trans, anon_map[row_idx], tipo_veiculo


def extract_sheet_data(ws_values, ws_formulas, sheet_name, month):
    sheet_data = []

    first_row = list(ws_values.iter_rows(min_row=1, max_row=1, values_only=True))[0]
    date_columns = {}
    for col_idx in range(2, len(first_row)):
        val = first_row[col_idx]
        if isinstance(val, datetime):
            date_columns[col_idx] = val

    if not date_columns:
        return sheet_data

    anon_map = {}
    row_idx = 2

    while row_idx <= ws_values.max_row:
        current_row = list(ws_values.iter_rows(
            min_row=row_idx, max_row=row_idx, values_only=True))[0]

        if not current_row or not current_row[0]:
            row_idx += 1
            continue

        col_b = str(current_row[1]).strip().upper() if current_row[1] else ""
        if col_b != "KM":
            row_idx += 1
            continue

        transportador, viatura, tipo_veiculo = parse_cell_a(
            current_row[0], sheet_name, row_idx, anon_map)

        if not viatura:
            row_idx += 1
            continue

        km_row  = current_row
        p_row   = list(ws_values.iter_rows(min_row=row_idx+1, max_row=row_idx+1, values_only=True))[0]
        vt_row  = list(ws_values.iter_rows(min_row=row_idx+2, max_row=row_idx+2, values_only=True))[0]
        sd_row  = list(ws_values.iter_rows(min_row=row_idx+3, max_row=row_idx+3, values_only=True))[0]
        tot_row = list(ws_values.iter_rows(min_row=row_idx+4, max_row=row_idx+4, values_only=True))[0]

        formula_cell = ws_formulas.cell(row=row_idx + 3, column=3).value
        tipo_formula, km_gratis, rate = parse_formula(formula_cell)

        for col_idx, date_obj in date_columns.items():
            if col_idx >= len(km_row):
                continue

            km_value = clean_value(km_row[col_idx])
            if not km_value or km_value == 0:
                continue

            portagens   = clean_value(p_row[col_idx]   if col_idx < len(p_row)   else None)
            valor_total = clean_value(vt_row[col_idx]  if col_idx < len(vt_row)  else None)
            sub_divisao = clean_value(sd_row[col_idx]  if col_idx < len(sd_row)  else None)
            total       = clean_value(tot_row[col_idx] if col_idx < len(tot_row) else None)

            sheet_data.append({
                'transportador': transportador,
                'tipo_veiculo':  tipo_veiculo,
                'viatura':       viatura,
                'data':          date_obj.date(),
                'dia':           date_obj.day,
                'mes':           month,
                'km':            int(km_value),
                'portagens':     round(portagens, 2)   if portagens   else 0.0,
                'preco_base':    round(valor_total, 2) if valor_total else None,
                'sub_divisao':   round(sub_divisao, 2) if sub_divisao else None,
                'tipo_contrato': tipo_formula,
                'km_gratis':     km_gratis,
                'rate_km':       rate,
                'total':         round(total, 2)        if total       else None,
            })

        row_idx += 5
        while row_idx <= ws_values.max_row:
            check = list(ws_values.iter_rows(
                min_row=row_idx, max_row=row_idx, values_only=True))[0]
            if check and check[0]:
                break
            row_idx += 1

    return sheet_data


# ── LEITURA ──────────────────────────────────────────────────────────────────
all_data = []

for month, filename in MONTH_FILES.items():
    filepath = os.path.join(BASE_PATH, filename)
    if not os.path.exists(filepath):
        print(f"Não encontrado: {filename}")
        continue
    print(f"A ler: {filename}")

    wb_val  = openpyxl.load_workbook(filepath, data_only=True)
    wb_form = openpyxl.load_workbook(filepath, data_only=False)

    for sheet_name in wb_val.sheetnames:
        ws_v = wb_val[sheet_name]
        ws_f = wb_form[sheet_name]
        data = extract_sheet_data(ws_v, ws_f, sheet_name.strip(), month)
        all_data.extend(data)
        if data:
            print(f"  {sheet_name}: {len(data)} registos")

    wb_val.close()
    wb_form.close()

df = pd.DataFrame(all_data)
df = df.sort_values(['data', 'viatura']).reset_index(drop=True)

print(f"\nTotal: {len(df)} registos | {df['viatura'].nunique()} viaturas")
df.head(20)


A ler: 01 JANEIRO.xlsx
  TRANSMAE: 22 registos
  TJA: 69 registos
  TPCF: 47 registos
  Florêncio & Silva: 219 registos
  Paulo Duarte: 22 registos
  Transaura: 40 registos
  Globalshield: 12 registos
A ler: 02 FEVEREIRO.xlsx
  TRANSMAE: 21 registos
  TJA: 69 registos
  TPCF: 39 registos
  Florêncio & Silva: 215 registos
  Paulo Duarte: 20 registos
  Transaura: 40 registos
  Globalshield: 19 registos
A ler: 03 MARÇO.xlsx
  TRANSMAE: 22 registos
  TJA: 67 registos
  TPCF: 46 registos
  Florêncio & Silva: 229 registos
  Paulo Duarte: 20 registos
  Transaura: 44 registos
  Globalshield: 17 registos
A ler: 04 ABRIL.xlsx
  TRANSMAE: 13 registos
  TJA: 84 registos
  TPCF: 47 registos
  Florêncio & Silva: 240 registos
  Paulo Duarte: 21 registos
  Transaura: 52 registos
  Globalshield: 18 registos
A ler: 05 MAIO.xlsx
  TJA: 102 registos
  TPCF: 48 registos
  Florêncio & Silva: 244 registos
  Paulo Duarte: 21 registos
  Transaura: 50 registos
  Globalshield: 19 registos
A ler: 06 JUNHO.xlsx
  

,transportador,tipo_veiculo,viatura,data,dia,mes,km,portagens,preco_base,sub_divisao,tipo_contrato,km_gratis,rate_km,total
0,TRANSMAE,NaN,18-SD-56,2024-01-02,2,1,52,32.0,230.24,NaN,NaN,NaN,NaN,NaN
1,Transaura,NaN,48-QQ-68,2024-01-02,2,1,176,0.0,225.00,110.88,fixo,0.0,0.63,335.88
2,Florêncio & Silva,NaN,66-RF-32,2024-01-02,2,1,30,0.0,246.00,18.00,fixo,0.0,0.60,264.00
3,TJA,NaN,69-SI-53,2024-01-02,2,1,282,0.0,235.00,135.36,fixo,0.0,0.48,370.36
4,Transaura,NaN,83-94-XB,2024-01-02,2,1,181,0.0,225.00,114.03,fixo,0.0,0.63,339.03
5,Globalshield,NaN,97-LF-43,2024-01-02,2,1,275,0.0,200.00,134.75,fixo,0.0,0.49,334.75
6,Paulo Duarte,CAPILAR,AA-35-ND,2024-01-02,2,1,227,0.0,200.00,102.15,fixo,0.0,0.45,302.15
7,TPCF,NaN,AN-59-UL,2024-01-02,2,1,204,0.0,234.00,1.44,excedente,200.0,0.36,235.44
8,Florêncio & Silva,ELÉTRICO,AQ-93-XZ,2024-01-02,2,1,131,0.0,214.00,20.96,fixo,0.0,0.16,234.96
9,Florêncio & Silva,NaN,AR-59-PH,2024-01-02,2,1,345,0.0,347.00,207.00,fixo,0.0,0.60,554.00


In [ ]:
import sqlite3
import platform

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    DB_PATH = "inform_27.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Criar tabela com nome dinâmico baseado no ANO
table_name = f"km_diario_{ANO}"

cursor.execute(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        transportador   TEXT,
        tipo_veiculo    TEXT,
        viatura         TEXT,
        data            DATE,
        dia             INTEGER,
        mes             INTEGER,
        km              INTEGER,
        portagens       REAL,
        preco_base      REAL,
        sub_divisao     REAL,
        tipo_contrato   TEXT,
        km_gratis       REAL,
        rate_km         REAL,
        total           REAL,
        criado_em       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()

df_insert = df.copy()
df_insert['data'] = df_insert['data'].astype(str)

df_insert.to_sql(
    name      = table_name,
    con       = conn,
    if_exists = 'append',
    index     = False,
)
conn.commit()

total = cursor.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
meses = cursor.execute(f"SELECT DISTINCT mes FROM {table_name} ORDER BY mes").fetchall()
trans = cursor.execute(f"SELECT transportador, COUNT(*) as n FROM {table_name} GROUP BY transportador ORDER BY n DESC").fetchall()

print(f"✅ Registos na tabela {table_name}: {total:,}")
print(f"📅 Meses: {[m[0] for m in meses]}")
print(f"\n📦 Por transportador:")
for t in trans:
    print(f"   {t[0]:<25} {t[1]:>5} registos")

conn.close()
print(f"\n✅ Concluído.")


✅ Registos na tabela km_diario_2025: 14,258
📅 Meses: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

📦 Por transportador:
   Florêncio & Silva          5794 registos
   TJA                        2868 registos
   TFS                        2106 registos
   CMTIR                      1314 registos
   Transaura                   988 registos
   TPCF                        692 registos
   Paulo Duarte                374 registos
   Ramitrans                    80 registos
   CMTIR EXTRA                  26 registos
   RAMITRANS                    16 registos

✅ Concluído.
